In [2]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from newspaper import Article
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler, OrdinalEncoder, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
import plotly.graph_objects as go
import html
import uuid
import joblib

Testing loading my model and making sure it works 

In [3]:
bundle = joblib.load("publisher_clustering_model.joblib")

publisher_cluster_cols = bundle["publisher_cluster_cols"]
publisher_scaler = bundle["scaler"]
publisher_kmeans = bundle["kmeans"]
publisher_pca = bundle["pca"]

print("Loaded saved model successfully.")
print("Features:", publisher_cluster_cols)
print("Number of clusters:", bundle["n_clusters"])
print("Original silhouette score:", bundle["silhouette_score"])

Loaded saved model successfully.
Features: ['overall_accuracy', 'overall_avg_sentiment', 'log_overall_articles', 'unique_tickers', 'overall_long_term_ratio']
Number of clusters: 4
Original silhouette score: 0.24774893839112128


Model loaded -> I am going to verify I can recreate the clustering using the loaded model

In [5]:
data = pd.read_csv("data_visualization.csv")

data = data.dropna(subset=["ticker", "publisher"])
data["publisher"] = data["publisher"].astype(str).str.strip()
data["ticker"] = data["ticker"].astype(str).str.strip()

numeric_cols = [
    "sentiment_encoded",
    "return_30d",
    "return_6m",
    "long_term",
    "accuracy"
]

for col in numeric_cols:
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors="coerce")

data["accuracy"] = data["accuracy"].fillna(0)
data["sentiment_encoded"] = data["sentiment_encoded"].fillna(0)
data["long_term"] = data["long_term"].fillna(0)

print("Shape:", data.shape)

Shape: (23908, 21)


In [6]:
publisher_overall_test = data.groupby("publisher").agg(
    overall_articles=("headline", "count"),
    overall_accuracy=("accuracy", "mean"),
    overall_avg_sentiment=("sentiment_encoded", "mean"),
    overall_long_term_ratio=("long_term", "mean"),
    unique_tickers=("ticker", "nunique")
).reset_index()

publisher_overall_test["log_overall_articles"] = np.log1p(
    publisher_overall_test["overall_articles"]
)

publisher_X_test = publisher_overall_test[publisher_cluster_cols].fillna(0)

print("Publisher feature table rebuilt.")
print(publisher_X_test.head())

Publisher feature table rebuilt.
   overall_accuracy  overall_avg_sentiment  log_overall_articles  \
0          0.517241              -0.103448              3.401197   
1          0.615385               0.015385              4.875197   
2          0.633803              -0.028169              4.276666   
3          0.580000              -0.180000              3.931826   
4          0.500739              -0.006401              7.616776   

   unique_tickers  overall_long_term_ratio  
0              21                 0.206897  
1              31                 0.069231  
2              23                 0.000000  
3              23                 0.200000  
4              42                 0.012309  


In [7]:
publisher_X_scaled_test = publisher_scaler.transform(publisher_X_test)
predicted_clusters = publisher_kmeans.predict(publisher_X_scaled_test)
pca_values_test = publisher_pca.transform(publisher_X_scaled_test)

publisher_overall_test["predicted_cluster"] = predicted_clusters
publisher_overall_test["pca1"] = pca_values_test[:, 0]
publisher_overall_test["pca2"] = pca_values_test[:, 1]

print("Predictions completed.")
publisher_overall_test.head()

Predictions completed.


,publisher,overall_articles,overall_accuracy,overall_avg_sentiment,overall_long_term_ratio,unique_tickers,log_overall_articles,predicted_cluster,pca1,pca2
0,Alex Furno,29,0.517241,-0.103448,0.206897,21,3.401197,2,-1.039399,-0.025834
1,Allie Wickman,130,0.615385,0.015385,0.069231,31,4.875197,1,0.849055,0.005999
2,Benzinga Newdesk,71,0.633803,-0.028169,0.000000,23,4.276666,3,0.072029,-0.136446
3,Benzinga News Desk,50,0.580000,-0.180000,0.200000,23,3.931826,3,-0.623368,0.185393
4,Benzinga Newsdesk,2031,0.500739,-0.006401,0.012309,42,7.616776,1,2.685610,-1.221715


In [8]:
loaded_labels = publisher_kmeans.labels_
predicted_labels = predicted_clusters

same_labels = np.array_equal(loaded_labels, predicted_labels)

print("Do loaded-model predictions exactly match original labels?", same_labels)
print("Number of publishers:", len(predicted_labels))

Do loaded-model predictions exactly match original labels? True
Number of publishers: 80


In [9]:
test_silhouette = silhouette_score(publisher_X_scaled_test, predicted_clusters)
print("Recomputed silhouette score:", test_silhouette)
print("Original silhouette score:", bundle["silhouette_score"])

Recomputed silhouette score: 0.24774893839112128
Original silhouette score: 0.24774893839112128
